# Create Swift Format Dataset

Supports nested folder structure:
```
data/
├── documents/
│   ├── aus_passport/
│   ├── aus_medicare_card/{green,blue,yellow}/
│   └── aus_driver_license/{act,nsw,...}/
└── labels/  (mirrors documents/)
```
`template.json` / `template.png` are automatically ignored.

## 1. Install


In [ ]:

print("Ready")

## 2 · Configuration

In [ ]:
from pathlib import Path

DATA_DIR   = Path("./data")                    # root data dir
DOC_DIR    = DATA_DIR / "documents"            # image/pdf
LABEL_DIR  = DATA_DIR / "labels"              # json label
OUTPUT_DIR = DATA_DIR / "swift_dataset"        # output Swift format
SCHEMA_DIR = Path("./schemas")                 # schema JSON (not used)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Document types to skip
SKIP_DOC_TYPES = {
    "aus_energy_bill",
    "act", "nsw", "nt", "qld", "sa", "tas", "wa",  # sub-state folders
}

TRAIN_RATIO   = 0.80   # train
DEV_RATIO     = 0.0    # validation (disabled — too few samples)
TEST_RATIO    = 0.2    # test
MIN_FOR_SPLIT = 3      # minimum samples to apply split
RANDOM_SEED   = 42

# ── System prompt (shared for train & inference) ───────────────────────────
SYSTEM_PROMPT = (
    "You are a document processing expert specialized in extracting structured "
    "information from official Australian identity and government documents. "
    "Always respond with valid JSON only, no extra text or explanation."
)

# ── Per-document-type hints ────────────────────────────────────────────────
# Used to inject context into the user prompt so the model knows:
#   - what kind of document it is looking at
#   - where key fields are typically located
#   - what format values should follow
# IMPORTANT: these same hints must be used at inference time to match training.
DOC_TYPE_HINTS = {
"AUS_DRIVER_LICENSE": (
    "This is an Australian Driver's Licence. "
    "- age_indicator: format DD-YY (birth day - birth year last 2 digits), e.g. '08-04'\n"
    "- barcode_number: starts with 'AB', 18+ chars, copy exactly as printed\n"
    "- dob_watermark: 8 digits DDMMYYYY with NO separators, e.g. '09092004'\n"
    "- full_name in front.holder: first_name + last_name only, no title\n"
    "- transport_notice: copy full text exactly, do not truncate\n"
),
"AUS_MEDICARE_CARD": (
    "This is an Australian Medicare Card (may be green, blue, or yellow). "
    "- card_type: exactly one of 'regular' (not show in card), 'interim card', 'reciprocal health care'\n"
    "- full_name: must include position number prefix.\n"
    "- middle_initial: single letter only, null if not present\n"
    "- expiry_date: return as YYYY-MM-DD regardless of how it appears on card\n"
),
"AUS_PASSPORT": (
    "This is an Australian Passport. "
    "- mrz_line1: first line of the MRZ (44 characters)."
    "- mrz_line2: second line of the MRZ, immediately below mrz_line1 (44 characters)."
    "- Both lines must contain exactly 44 characters."
    "- Use '<' as the filler character when padding is required."
    "- If the extracted line exceeds 44 characters, keep only the first 44 characters."
    "- issuing_country: always 'AUS', never 'Australia'\n"
),
"_default": (
    "This is an official Australian identity or government document. "
    "Extract all requested fields exactly as they appear. "
    "Return dates in YYYY-MM-DD format and numbers without spaces or separators."
)
}

# ── User prompt template ───────────────────────────────────────────────────
# Placeholders:
#   {image_tokens}  — one <image> per page/image
#   {doc_hint}      — per-doc-type description from DOC_TYPE_HINTS

USER_PROMPT_TEMPLATE = (
    "Document image: {image_tokens}\n\n"
    "Context: {doc_hint}\n\n"
    "Extract all fields and return ONLY valid JSON matching this structure:\n"
    "{schema}\n\n"
    "Rules:\n"
    "- null for missing fields, do NOT omit keys\n"
    "- Dates: YYYY-MM-DD\n"
    "- No markdown, no explanation"
)

print("Configuration loaded.")
print(f"Split: TRAIN={TRAIN_RATIO:.0%}  DEV={DEV_RATIO:.0%}  TEST={TEST_RATIO:.0%}")
print(f"Doc type hints defined: {len(DOC_TYPE_HINTS) - 1} types + 1 default")


## 3 · Load Data

In [ ]:
import json
import mimetypes
import random
import io
from collections import defaultdict, Counter
from pathlib import Path

import pandas as pd
from PIL import Image
import pypdfium2 as pdfium
from tqdm import tqdm


def find_image_for_label(label_path: Path, doc_dir: Path) -> Path | None:
    """Find PDF/img matching with file label JSON."""
    rel_path = label_path.relative_to(LABEL_DIR)
    candidate_extensions = [".jpg", ".jpeg", ".png", ".JPG", ".PNG", ".pdf"]
    for ext in candidate_extensions:
        candidate = doc_dir / rel_path.parent / (label_path.stem + ext)
        if candidate.exists():
            return candidate
    return None


def build_record(label_path: Path, image_path: Path) -> dict:
    """Create a record from (label, image)."""
    rel_path = label_path.relative_to(LABEL_DIR)
    return {
        "filename"   : image_path.name,
        "rel_path"   : str(rel_path),
        "doc_type"   : "/".join(rel_path.parts[:-1]),
        "filetype"   : mimetypes.guess_type(image_path)[0] or "image/jpeg",
        "target_data": json.dumps(
            json.load(open(label_path, encoding="utf-8")),
            ensure_ascii=False
        ),
        "doc_bytes"  : image_path.read_bytes(),
    }


def load_all_records(doc_dir: Path, label_dir: Path, skip_types: set) -> list[dict]:
    """Load all records from label dir and img dir"""
    label_files = [
        p for p in sorted(label_dir.rglob("*.json"))
        if p.stem != "template"
        and not any(part in skip_types for part in p.parts)
    ]
    print(f"Found {len(label_files)} file label")

    records = []
    for label_path in tqdm(label_files, desc="Loading"):
        image_path = find_image_for_label(label_path, doc_dir)
        if image_path is None:
            print(f"  SKIP: {label_path.relative_to(label_dir)}")
            continue
        records.append(build_record(label_path, image_path))

    print(f"Đã load {len(records)} records")
    return records


all_records = load_all_records(DOC_DIR, LABEL_DIR, SKIP_DOC_TYPES)

In [ ]:

df_info = pd.DataFrame([
    {k: v for k, v in r.items() if k != "doc_bytes"}
    for r in all_records
])
distribution = df_info["doc_type"].value_counts().sort_index()
print(distribution.to_string())
print(f"\nTổng: {len(df_info)} mẫu, {len(distribution)} loại tài liệu")

## 4 · Split

In [ ]:
random.seed(RANDOM_SEED)


def compute_split_sizes(n: int, dev_ratio: float, test_ratio: float) -> tuple[int, int, int]:
    """Tính số lượng mẫu cho mỗi split."""
    n_test  = int(n * test_ratio)
    n_dev   = int(n * dev_ratio)
    n_train = n - n_dev - n_test
    return n_train, n_dev, n_test


def split_records_by_type(
    records: list[dict],
    dev_ratio: float,
    test_ratio: float,
    min_for_split: int,
) -> tuple[list, list, list]:
    """Chia records thành train/dev/test theo từng loại tài liệu."""
    by_type: dict[str, list] = defaultdict(list)
    for record in records:
        by_type[record["doc_type"]].append(record)

    train_records, dev_records, test_records = [], [], []

    header = f"{'Document Type':<42} {'N':>5} {'Train':>6} {'Dev':>5} {'Test':>5}"
    print(header)
    print("-" * 65)

    for doc_type in sorted(by_type):
        type_records = by_type[doc_type]
        random.shuffle(type_records)
        n = len(type_records)

        if n >= min_for_split:
            n_train, n_dev, n_test = compute_split_sizes(n, dev_ratio, test_ratio)
        else:
            n_train, n_dev, n_test = n, 0, 0  # quá ít → giữ hết trong train

        train_records.extend(type_records[:n_train])
        dev_records.extend(type_records[n_train : n_train + n_dev])
        test_records.extend(type_records[n_train + n_dev :])

        note = "  *" if n < min_for_split else ""
        print(f"{doc_type:<42} {n:>5} {n_train:>6} {n_dev:>5} {n_test:>5}{note}")

    total = len(records)
    print("-" * 65)
    print(
        f"{'TOTAL':<42} {total:>5} "
        f"{len(train_records):>6} {len(dev_records):>5} {len(test_records):>5}"
    )
    return train_records, dev_records, test_records


train_records, dev_records, test_records = split_records_by_type(
    all_records, DEV_RATIO, TEST_RATIO, MIN_FOR_SPLIT
)

## 5 · Extract Images & Build Swift Format

In [ ]:
def extract_images_from_record(record: dict, max_pages: int = 4) -> list[str]:
    """Extract images/pages from a record (supports PDF and raster images)."""
    images_dir = OUTPUT_DIR / "images"
    images_dir.mkdir(exist_ok=True)

    stem     = Path(record["filename"]).stem
    filetype = record["filetype"]
    image_paths = []

    try:
        if filetype == "application/pdf":
            image_paths = _extract_pdf_pages(record["doc_bytes"], stem, images_dir, max_pages)
        else:
            image_paths = _save_image(record["doc_bytes"], stem, images_dir)
    except Exception as exc:
        print(f"  Error processing {stem}: {exc}")

    return image_paths


def _extract_pdf_pages(pdf_bytes: bytes, stem: str, images_dir: Path, max_pages: int) -> list[str]:
    """Render PDF pages to PNG, return list of absolute paths."""
    pdf = pdfium.PdfDocument(pdf_bytes)
    paths = []
    for page_idx in range(min(len(pdf), max_pages)):
        out_path = images_dir / f"{stem}_page{page_idx:03}.png"
        if not out_path.exists():
            pdf[page_idx].render(scale=1.53).to_pil().save(out_path)
        paths.append(str(out_path.resolve()))
    return paths


def _save_image(img_bytes: bytes, stem: str, images_dir: Path) -> list[str]:
    """Save image as PNG, return list with 1 path."""
    out_path = images_dir / f"{stem}.png"
    if not out_path.exists():
        Image.open(io.BytesIO(img_bytes)).save(out_path)
    return [str(out_path.resolve())]


def get_doc_hint(doc_type: str) -> str:
    """
    Match doc_type (extracted from folder path) to DOC_TYPE_HINTS.
    Path format:  "aus_driver_license/vic"
    Dict keys:    "AUS_DRIVER_LICENSE", "aus_driver_license/vic", "_default"
    Lookup order:
      1. Exact match (lowercase)           e.g. "aus_driver_license/vic"
      2. Exact match (uppercase)           e.g. "AUS_DRIVER_LICENSE"
      3. Parent folder (uppercase)         e.g. "AUS_DRIVER_LICENSE" from "aus_driver_license/vic"
      4. _default fallback
    """
    # Exact lowercase
    if doc_type in DOC_TYPE_HINTS:
        return DOC_TYPE_HINTS[doc_type]

    # Exact uppercase
    key_upper = doc_type.upper().replace("/", "_")  # "aus_driver_license/vic" → "AUS_DRIVER_LICENSE_VIC"
    # Also try just uppercasing: "aus_driver_license" → "AUS_DRIVER_LICENSE"
    key_upper_base = doc_type.split("/")[0].upper()   # "AUS_DRIVER_LICENSE"
    
    if key_upper in DOC_TYPE_HINTS:
        return DOC_TYPE_HINTS[key_upper]
    if key_upper_base in DOC_TYPE_HINTS:
        return DOC_TYPE_HINTS[key_upper_base]
    return DOC_TYPE_HINTS["_default"]



def build_user_prompt(image_paths: list[str], label_dict: dict, doc_type: str) -> str:
    """
    Build the user prompt for a given document using JSON schema format.
    """
    image_tokens = "<image>" * len(image_paths)
    doc_hint     = get_doc_hint(doc_type)
    
    # Schema: keys only, all values -> null
    schema_keys  = {k: None for k in label_dict.keys()}
    schema       = json.dumps(schema_keys, indent=2, ensure_ascii=False)
    
    return USER_PROMPT_TEMPLATE.format(
        image_tokens=image_tokens,
        doc_hint=doc_hint,
        schema=schema
    )

def flatten_label(d: dict, parent_key: str = "", sep: str = "_") -> dict:
    result = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            result.update(flatten_label(v, new_key, sep))
        else:
 
            if isinstance(v, str) and v.startswith("[") and v.endswith("]"):
                try:
                    v = json.loads(v)
                except json.JSONDecodeError:
                    pass
            result[new_key] = v
    return result

def build_swift_sample(record: dict) -> dict | None:
    """
     Swift Format =  Ground Truth + Json example
    """
    image_paths = extract_images_from_record(record)
    if not image_paths:
        return None

    label_dict = json.loads(record["target_data"])
    doc_type   = record["doc_type"]
    
    flat_label = flatten_label(label_dict)
    
    user_prompt = build_user_prompt(image_paths, flat_label, doc_type)
    
    assistant_response = json.dumps(flat_label, ensure_ascii=False)

    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_prompt},
            {"role": "assistant", "content": assistant_response},
        ],
        "images": image_paths,
    }
# ── Preview: fields per doc type ──────────────────────────────────────────
keys_by_doctype: dict[str, set] = defaultdict(set)
for r in all_records:
    label = json.loads(r["target_data"])
    keys_by_doctype[label.get("document_type", "UNKNOWN")].update(label.keys())

print("Fields for each doc type:")
for dtype, keys in sorted(keys_by_doctype.items()):
    print(f"  {dtype}: {sorted(keys)}")

# ── Preview: example prompt per doc type ──────────────────────────────────
print("" + "=" * 60)
print("EXAMPLE PROMPTS PER DOC TYPE")
print("=" * 60)
seen_types = set()
for r in all_records:
    doc_type = r["doc_type"]
    if doc_type in seen_types:
        continue
    seen_types.add(doc_type)
    label = json.loads(r["target_data"])
    prompt = build_user_prompt(["<image>"], label, doc_type)
    print(f"--- {doc_type} ---")
    print(f"[SYSTEM]{SYSTEM_PROMPT}")
    print(f"[USER]{prompt}")
    print()


## 6 · Save Swift Format Files

In [ ]:
def save_swift_split(records: list[dict], split_name: str) -> Path:
    """
    Create Swift samples from record list và save to file JSON.
    return file output.
    """
    samples = []
    skipped = 0

    for record in tqdm(records, desc=split_name):
        sample = build_swift_sample(record)
        if sample:
            samples.append(sample)
        else:
            skipped += 1

    output_path = OUTPUT_DIR / f"conversations_{split_name}_swift_format.json"
    json.dump(samples, open(output_path, "w", encoding="utf-8"), indent=2, ensure_ascii=False)

    skip_note = f"  ({skipped} skipped)" if skipped else ""
    print(f"  {split_name}: {len(samples)} sample → {output_path.name}{skip_note}")
    return output_path


output_train = save_swift_split(train_records, "train")
output_dev   = save_swift_split(dev_records,   "dev")
output_test  = save_swift_split(test_records,  "test")

In [ ]:
def show_split_distribution(output_path: Path, split_name: str) -> None:
    """print split."""
    data = json.load(open(output_path))
    doc_types = [
        json.loads(sample["messages"][2]["content"]).get("document_type", "?")
        for sample in data
    ]
    print(f"{split_name} ({len(data)} mẫu):")
    for doc_type, count in sorted(Counter(doc_types).items()):
        print(f"  {doc_type}: {count}")


show_split_distribution(output_train, "TRAIN")
print()
show_split_distribution(output_dev, "DEV")
print()
show_split_distribution(output_test, "TEST")

In [ ]:
# check sample train
train_data = json.load(open(output_train))
if train_data:
    first_sample = train_data[0]
    print("=== Kiểm tra sample đầu tiên ===")
    print(f"Images : {first_sample['images']}")
    print(f"\n[SYSTEM]\n{first_sample['messages'][0]['content']}")
    print(f"\n[USER]\n{first_sample['messages'][1]['content']}")
    print(f"\n[ASSISTANT] (100 ký tự đầu)\n{first_sample['messages'][2]['content'][:100]}...")

In [ ]:

print("File output:")
for file_path in sorted(OUTPUT_DIR.iterdir()):
    if file_path.is_file():
        size_kb = file_path.stat().st_size / 1024
        print(f"  {file_path.name:<50} {size_kb:>8.1f} KB")

image_count = len(list((OUTPUT_DIR / "images").glob("*")))
print(f"\nSố ảnh: {image_count} files")

## Done

### Training config
```python
train_dataset = "./data/swift_dataset/conversations_train_swift_format.json"
val_dataset   = "./data/swift_dataset/conversations_dev_swift_format.json"
test_dataset  = "./data/swift_dataset/conversations_test_swift_format.json"
```

### Adding a new document type
1. Add images → `documents/<type>/<subtype>/`
2. Add labels → `labels/<type>/<subtype>/`
3. Add schema + update `label_to_schema_mapping.json`
4. Re-run notebook